# 00 — Python Data Model and Special Methods

Explore how built-in syntax maps to special methods and how to design predictable Pythonic value objects and collections.

*Advanced · 90–150 min*

## Objectives
- Understand protocols for len, iteration, membership and repr
- Implement meaningful equality and hashing
- Explain the difference between identity and equality
- Build consistent ordering with NotImplemented

## Examples

### 1. Collection protocols

A collection-like object can expose length, iteration, and membership through special methods. Using a tuple internally prevents external code from mutating its items by accident. Built-ins dispatch to the object's type.

In [ ]:
class Basket:
    def __init__(self, items):
        self._items = tuple(items)
    def __len__(self):
        return len(self._items)
    def __iter__(self):
        return iter(self._items)
    def __contains__(self, value):
        return value in self._items
    def __repr__(self):
        return f"Basket({self._items!r})"

basket = Basket(["book", "pen"])
print(basket, len(basket), list(basket), "pen" in basket)
assert len(basket) == 2

**Design question:** Should a consumer expect Basket to implement append? Why or why not?

### 2. Boolean and string conversion

When present, __bool__ controls truth testing; otherwise Python can use __len__. The result of __bool__ must be a boolean. __str__ is user-oriented, while __repr__ usually helps debugging. Neither should reveal secrets.

In [ ]:
class SearchResult:
    def __init__(self, rows):
        self._rows = list(rows)
    def __bool__(self):
        return bool(self._rows)
    def __str__(self):
        return f"{len(self._rows)} rows"
    def __repr__(self):
        return f"SearchResult(count={len(self._rows)})"

empty = SearchResult([])
nonempty = SearchResult([1, 2])
print(bool(empty), bool(nonempty), str(nonempty), repr(nonempty))

**Design question:** Why is an object that always returns True from __bool__ potentially misleading?

### 3. Value equality and hashability

Equal values must produce equal hashes for dictionary and set consistency. A frozen dataclass with immutable fields is a convenient key. Hashing mutable fields can break lookup after mutation.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Coordinate:
    x: int
    y: int

a, b = Coordinate(2, 3), Coordinate(2, 3)
locations = {a: "office"}
print(a is b, a == b, locations[b])
assert hash(a) == hash(b)

**Design question:** What happens if a frozen dataclass contains an unhashable list?

### 4. Ordering and unsupported operands

Return NotImplemented for unsupported comparisons, allowing Python to try reflected operations. functools.total_ordering derives the remaining ordering operators from equality and one comparison, but only if the ordering is conceptually sound.

In [ ]:
from functools import total_ordering

@total_ordering
class Rank:
    def __init__(self, number):
        self.number = number
    def __eq__(self, other):
        if not isinstance(other, Rank):
            return NotImplemented
        return self.number == other.number
    def __lt__(self, other):
        if not isinstance(other, Rank):
            return NotImplemented
        return self.number < other.number

print([r.number for r in sorted([Rank(3), Rank(1), Rank(2)])])

**Design question:** What should be done if two ranks are incomparable rather than ordered?

## Try it (optional)

Small open experiments to explore the topic. The **graded** exercises for this topic, with fill-in-the-blank code and automatic checks, are in the [workbook](../exercises/01_internals_typing_architecture_exercises.ipynb).

**Exercise 1:** Implement immutable Money with currency-sensitive equality and display.

In [ ]:
# Exercise 1
# TODO: Implement, demonstrate, and add assertions.

**Exercise 2:** Build an object with len/iter/contains and test it with an empty collection.

In [ ]:
# Exercise 2
# TODO: Implement, demonstrate, and add assertions.

**Exercise 3:** Demonstrate hash compatibility for distinct equal dataclass objects.

In [ ]:
# Exercise 3
# TODO: Implement, demonstrate, and add assertions.

**Exercise 4:** Implement a consistent task priority ordering and test ties.

In [ ]:
# Exercise 4
# TODO: Implement, demonstrate, and add assertions.

## Common mistakes

- Do not equate `is` with `==`.
- __hash__ must agree with equality and stable object state.
- Special methods should satisfy users' normal expectations.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** If you define __eq__, you should also think about…

   - **a)** __len__
   - **b)** __iter__
   - **c)** __hash__

**2.** What should __lt__ return for an unsupported type?

   - **a)** raise TypeError directly
   - **b)** NotImplemented
   - **c)** False

**3.** `a is b` checks…

   - **a)** The same object
   - **b)** The same type
   - **c)** Equal values

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/00', {1: '21b43ee94f7f', 2: 'c5dcf34c1dfb', 3: 'b43d4be3104b'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [01 — Descriptors, Properties, and Metaclasses](01_descriptors_and_metaclasses.ipynb)